# Formal V11 Symbolic Regression: Iteration 4

This notebook runs one complete frozen development rotation:

1. V10 signed staged residual symbolic search, using the Iteration-4
   119 training cases and 595,000 stratified discovery rows;
2. V11 case-mean calibration and tail-aware localised symbolic search;
3. validation-only scalar gain selection, followed by one internal-test audit.

Every complete validation and internal-test case is evaluated. The 50 final
test case names are inventoried but their element files are not read. Each
PySR stage is segmented and recoverable, so a rerun resumes completed work.


## 1. Imports and package root


In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print


def resolve_package_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "src" / "formal_v11_four_rotation.py").exists():
            return candidate
    raise FileNotFoundError("Could not locate the NotebookCT3 package root.")


PACKAGE_ROOT = resolve_package_root()
if str(PACKAGE_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PACKAGE_ROOT / "src"))

print("Package root:", PACKAGE_ROOT)
print("Python:", sys.executable)


## 2. Run or resume the locked rotation


In [ ]:
from formal_v11_four_rotation import output_directory, run_formal_rotation

ITERATION = 4
RUN_FORMAL_ROTATION = True
OUTPUT_DIR = output_directory(PACKAGE_ROOT, ITERATION)

if RUN_FORMAL_ROTATION:
    RESULT = run_formal_rotation(PACKAGE_ROOT, ITERATION)
    display(pd.DataFrame([RESULT]))
else:
    RESULT = None
    print("Formal rotation skipped.")


## 3. Saved evidence


In [ ]:
roots = {
    "pipeline": OUTPUT_DIR,
    "v10": PACKAGE_ROOT / "outputs/10_signed_staged_residual_symbolic_pilot/iteration_4",
    "v11": PACKAGE_ROOT / "outputs/11_tail_aware_localised_symbolic/iteration_4",
    "gain": PACKAGE_ROOT / "outputs/12_v11_gain_stability_audit/iteration_4",
}

artifacts = {
    "pipeline_completion": roots["pipeline"] / "pipeline_complete.json",
    "v10_completion": roots["v10"] / "pilot_complete.json",
    "v11_completion": roots["v11"] / "pilot_complete.json",
    "gain_completion": roots["gain"] / "audit_complete.json",
    "selected_gain": roots["gain"] / "selected_gain.json",
    "selected_formula": roots["gain"] / "selected_gain_formula.txt",
    "split_metrics": roots["gain"] / "selected_models_split_metrics.csv",
}
for name, artifact in artifacts.items():
    print(f"{name:22s} exists={artifact.exists()}  {artifact}")

if artifacts["split_metrics"].exists():
    display(pd.read_csv(artifacts["split_metrics"]))
if artifacts["selected_formula"].exists():
    print("\n" + artifacts["selected_formula"].read_text(encoding="utf-8"))


## Interpretation

This is a development-rotation result, not external validation. Formula
coefficients are expected to change because the training cases change. The
primary questions are whether the same hierarchical method remains useful,
whether selected gain and feature families are reasonably stable, and whether
validation improvements survive the one-time internal-test audit.
